# Aula: K-Means em Ciência de Dados — agrupando músicas do Spotify

**Disciplina:** ICD-004 Análise de Algoritmos · Fatec Jundiaí
**Professor:** Prof. Humberto Zanetti
**Data:** 24/11/2026

**Integrantes:**
- Wellington Torres
<!-- Novos integrantes: acrescente uma linha "- Nome completo" acima. -->

**Repositório:** [github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify)

> Como usar esta aula: leia de cima para baixo e execute as células em ordem
> (*Ambiente de execução → Executar tudo*). As células **🧪 Experimente** convidam você a mudar
> um parâmetro e rodar de novo.

## Sumário

- [⚙️ Configuração do ambiente](#⚙️-Configuração-do-ambiente)
- [I. Introdução e Contextualização](#I.-Introdução-e-Contextualização)
  - [I.1 Aprendizado não supervisionado](#I.1-Aprendizado-não-supervisionado)
  - [I.2 Clustering (agrupamento)](#I.2-Clustering-(agrupamento))
  - [I.3 Como o K-Means funciona — implementado do zero](#I.3-Como-o-K-Means-funciona-—-implementado-do-zero)
  - [I.4 Quanto custa? Complexidade do K-Means](#I.4-Quanto-custa?-Complexidade-do-K-Means)
  - [I.5 Como avaliar: inércia, Método do Cotovelo e silhueta](#I.5-Como-avaliar:-inércia,-Método-do-Cotovelo-e-silhueta)
- [II. Ferramentas de Trabalho (Scikit-learn e Python)](#II.-Ferramentas-de-Trabalho-(Scikit-learn-e-Python))
  - [II.1 Bibliotecas e importações](#II.1-Bibliotecas-e-importações)
  - [II.2 O KMeans do scikit-learn](#II.2-O-KMeans-do-scikit-learn)
- [III. Escolha do Tema e Preparação dos Dados](#III.-Escolha-do-Tema-e-Preparação-dos-Dados)
  - [III.1 Tema: por que agrupar músicas?](#III.1-Tema:-por-que-agrupar-músicas?)
  - [III.2 O dataset: origem e dicionário de variáveis](#III.2-O-dataset:-origem-e-dicionário-de-variáveis)
  - [III.3 Carregamento e limpeza](#III.3-Carregamento-e-limpeza)
  - [III.4 Escolha das features](#III.4-Escolha-das-features)
  - [III.5 Padronização com StandardScaler](#III.5-Padronização-com-StandardScaler)
- [IV. Implementação e Aplicação Prática](#IV.-Implementação-e-Aplicação-Prática)
  - [IV.1 Definindo k: Método do Cotovelo (e silhueta)](#IV.1-Definindo-k:-Método-do-Cotovelo-(e-silhueta))
  - [IV.2 Treinando o K-Means](#IV.2-Treinando-o-K-Means)
  - [IV.3 Visualizando a separação dos clusters](#IV.3-Visualizando-a-separação-dos-clusters)
  - [IV.4 Centroides em relação aos pontos](#IV.4-Centroides-em-relação-aos-pontos)
- [V. Análise e conclusão](#V.-Análise-e-conclusão)
  - [V.1 Perfil de cada cluster](#V.1-Perfil-de-cada-cluster)
  - [V.2 Dando nome aos grupos](#V.2-Dando-nome-aos-grupos)
  - [V.3 Clusters × gêneros (com destaque para a música brasileira)](#V.3-Clusters-×-gêneros-(com-destaque-para-a-música-brasileira))
  - [V.4 Limitações do K-Means](#V.4-Limitações-do-K-Means)
  - [V.5 Próximos passos](#V.5-Próximos-passos)
- [Referências](#Referências)

> No Colab, o painel **Índice** (ícone ☰ à esquerda) também mostra esta estrutura.

## ⚙️ Configuração do ambiente

Esta célula prepara tudo o que o resto da aula usa. Ela fica aqui no começo, e não na Seção II,
porque a Seção I já desenha gráficos e roda um K-Means de exemplo. A Seção II explica cada
biblioteca com calma.

O que ela faz:

1. **Importa as bibliotecas**: `numpy` e `pandas` (dados), `matplotlib` e `seaborn` (gráficos) e
   as partes do `scikit-learn` usadas na aula (padronização, K-Means, métricas, PCA e dados de
   exemplo).
2. **Fixa a semente aleatória** `RANDOM_STATE = 42`. O K-Means e as amostragens sorteiam números;
   com a mesma semente, quem rodar a aula de novo vê **exatamente** os mesmos resultados.
3. **Define as constantes da análise**: o endereço do dataset, as 9 *features* de áudio usadas
   no agrupamento, os gêneros que vamos destacar e o tamanho das amostras usadas nas partes mais
   caras (silhueta e gráficos de dispersão).
4. **Escolhe o estilo dos gráficos** e **imprime as versões** do Python e das bibliotecas, para
   que dê para comparar se algum resultado sair diferente.

In [ ]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

# Semente única de toda a aula (constituição IV): não crie outra.
RANDOM_STATE = 42

# Dataset público no Hugging Face (lido direto da internet, sem upload).
URL_DATASET = (
    "https://huggingface.co/datasets/maharshipandya/spotify-tracks-dataset"
    "/resolve/main/dataset.csv"
)

# As 9 features contínuas de áudio que entram no K-Means (FR-035).
FEATURES = [
    "danceability", "energy", "loudness", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
]

# Gêneros em destaque na Seção V (FR-034) e gêneros internacionais de contraste.
GENEROS_BR = ["brazil", "mpb", "sertanejo", "pagode", "samba", "forro", "funk"]
GENEROS_CONTRASTE = ["classical", "metal", "sleep", "hip-hop", "edm", "comedy"]

# Amostras para as etapas caras (research R3).
TAMANHO_AMOSTRA_SILHUETA = 10_000  # a silhueta exata é O(n²)
TAMANHO_AMOSTRA_GRAFICOS = 6_000   # dispersão legível, sem virar um borrão

# Estilo padrão dos gráficos.
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams.update({"figure.figsize": (9, 5), "figure.dpi": 100, "axes.titleweight": "bold"})

print(f"Python       {sys.version.split()[0]}")
for nome, modulo in [("pandas", pd), ("numpy", np), ("scikit-learn", sklearn),
                     ("matplotlib", matplotlib), ("seaborn", sns)]:
    print(f"{nome:<12} {modulo.__version__}")

Acima estão as versões usadas nesta execução. No Colab elas mudam de tempos em tempos; se algum
número da aula sair um pouco diferente do texto, compare primeiro estas versões.

## I. Introdução e Contextualização

### I.1 Aprendizado não supervisionado

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.2 Clustering (agrupamento)

> 🚧 **Em construção** — ver T021 ([#9](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/9)).

### I.3 Como o K-Means funciona — implementado do zero

> 🚧 **Em construção** — ver T022 ([#10](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/10)), T023 ([#11](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/11)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T024 ([#12](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/12)).

Parâmetro a mudar: `K_BRINQUEDO` e `SEMENTE_INICIAL`. O que observar: inicializações diferentes mudam as iterações e às vezes o resultado (mínimo local).

### I.4 Quanto custa? Complexidade do K-Means

> 🚧 **Em construção** — ver T025 ([#13](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/13)).

### I.5 Como avaliar: inércia, Método do Cotovelo e silhueta

> 🚧 **Em construção** — ver T026 ([#14](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/14)).

## II. Ferramentas de Trabalho (Scikit-learn e Python)

### II.1 Bibliotecas e importações

> 🚧 **Em construção** — ver T033 ([#16](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/16)).

### II.2 O KMeans do scikit-learn

> 🚧 **Em construção** — ver T034 ([#17](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/17)).

## III. Escolha do Tema e Preparação dos Dados

### III.1 Tema: por que agrupar músicas?

> 🚧 **Em construção** — ver T007 ([#19](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/19)).

### III.2 O dataset: origem e dicionário de variáveis

> 🚧 **Em construção** — ver T008 ([#20](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/20)).

### III.3 Carregamento e limpeza

> 🚧 **Em construção** — ver T009 ([#21](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/21)), T010 ([#22](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/22)).

### III.4 Escolha das features

> 🚧 **Em construção** — ver T011 ([#23](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/23)).

### III.5 Padronização com StandardScaler

> 🚧 **Em construção** — ver T012 ([#24](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/24)).

## IV. Implementação e Aplicação Prática

### IV.1 Definindo k: Método do Cotovelo (e silhueta)

> 🚧 **Em construção** — ver T013 ([#26](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/26)), T014 ([#27](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/27)), T015 ([#28](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/28)).

### IV.2 Treinando o K-Means

> 🚧 **Em construção** — ver T016 ([#29](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/29)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T017 ([#30](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/30)).

Parâmetro a mudar: `K_TESTE`. O que observar: como os perfis se dividem ou se fundem com mais ou menos clusters.

### IV.3 Visualizando a separação dos clusters

> 🚧 **Em construção** — ver T018 ([#31](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/31)).

### IV.4 Centroides em relação aos pontos

> 🚧 **Em construção** — ver T019 ([#32](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/32)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T020 ([#33](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/33)).

Parâmetro a mudar: `USAR_PADRONIZACAO = True/False`. O que observar: sem padronizar, `tempo` e `loudness` dominam a distância e os clusters mudam.

## V. Análise e conclusão

### V.1 Perfil de cada cluster

> 🚧 **Em construção** — ver T027 ([#35](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/35)).

### V.2 Dando nome aos grupos

> 🚧 **Em construção** — ver T028 ([#36](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/36)).

### V.3 Clusters × gêneros (com destaque para a música brasileira)

> 🚧 **Em construção** — ver T029 ([#37](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/37)), T030 ([#38](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/38)).

#### 🧪 Experimente

> 🚧 **Em construção** — ver T031 ([#39](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/39)).

Parâmetro a mudar: `GENEROS_EXTRA`. O que observar: onde caem os gêneros que você escolher.

### V.4 Limitações do K-Means

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

### V.5 Próximos passos

> 🚧 **Em construção** — ver T032 ([#40](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/40)).

## Referências

> 🚧 **Em construção** — ver T035 ([#42](https://github.com/DataScience-Fatec-Projects/kmeans-musicas-spotify/issues/42)).